**Выгрузка численности населения с сайта Росстата Иркутской области**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

In [3]:
pip install requests beautifulsoup4 pandas openpyxl lxml

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import BytesIO
import re

In [5]:
# URL страницы
url = 'https://38.rosstat.gov.ru/folder/167937'

try:
    # Получаем содержимое страницы
    response = requests.get(url, timeout=30, verify=False)  # без сертификата
    response.raise_for_status()
    
    # Парсим HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Ищем ссылку на файл с нужным названием
    target_file = None
    for link in soup.find_all('a', href=True):

        link_text = link.get_text(strip=True)
        href = link['href']
        
        # Проверяем, содержит ли текст ссылки нужное название
        #file_name = 'Численность мужчин и женщин'
        file_name = 'Иркутская область_2012_2021_с учетом итогов Всероссийской переписи населения 2020 года'
        
        if file_name in href:
            # Формируем полный URL
            if href.startswith('http'):
                target_file = href
            elif href.startswith('/'):
                # Если ссылка относительная, добавляем домен
                base_url = 'https://38.rosstat.gov.ru'
                target_file = base_url + href
            else:
                # Если ссылка относительно текущей страницы
                target_file = url.rstrip('/') + '/' + href
            break
    
    if not target_file:
        raise FileNotFoundError("Файл с указанным названием не найден на странице")
    
    print(f"Найден файл: {target_file}")
    
    # Загружаем Excel‑файл
    file_response = requests.get(target_file, timeout=30, verify=False) # без сертификата
    file_response.raise_for_status()
    
    # Создаём объект BytesIO из содержимого файла
    excel_file = BytesIO(file_response.content)
    
    # Читаем Excel в датасет
    # df = pd.read_excel(excel_file)
    
    years = [2016, 2017, 2018, 2019]
    df_population = {}
    
    for year in years:
        page_name = f'на 01.01.{year}'
        # display(page_name)
        df_population[year] = pd.read_excel(excel_file, engine='openpyxl', sheet_name = page_name)
        # df_population[year] = df_population[year].dropna(how='all', axis=1)
        # df_population[year] = df_population[year].dropna(how='all')
        df_population[year] = df_population[year].iloc[7:108]
        df_population[year] = df_population[year].iloc[:,:5]
        df_population[year] =  df_population[year].reset_index(drop=True)
        df_population[year].iloc[100,0] = 100

    
    print("Файл успешно загружен в датасет!")
#    print(f"Размер датасета: {df.shape}")
#    print("\nПервые 5 строк:")
#    print(df.head())
    
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке страницы или файла: {e}")
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"Ошибка при обработке файла: {e}")

C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Найден файл: https://38.rosstat.gov.ru/storage/mediabank/Иркутская область_2012_2021_с учетом итогов Всероссийской переписи населения 2020 года.xlsx


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Файл успешно загружен в датасет!


In [6]:
# year = 2016
df = {}

columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]

data = []

for year in years:
    for col_number in range(2, df_population[year].shape[1]):
    
        new_row = []    
        new_row.append(year) # Год
        if (col_number == 2): # Пол
            new_row.append('всего')
        if (col_number == 3):
            new_row.append('мужчины')
        if (col_number == 4):
            new_row.append('женщины')

        #Всего
        value = 0
        for row_number in range(0, df_population[year].shape[0]):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
    
        # 0-4
        value = 0
        for row_number in range(0, 5):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 5-6
        value = 0
        for row_number in range(5, 7): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 7-14
        value = 0
        for row_number in range(7, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
      
        # 15-17
        value = 0
        for row_number in range(15, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 18-24
        value = 0
        for row_number in range(18, 25): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 25-34
        value = 0
        for row_number in range(25, 35): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 35-44
        value = 0
        for row_number in range(35, 44): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 45-54
        value = 0
        for row_number in range(45, 54): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 55-64
        value = 0
        for row_number in range(55, 64): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 65+
        value = 0
        for row_number in range(65, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-14
        value = 0
        for row_number in range(0, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 15+
        value = 0
        for row_number in range(15, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-17
        value = 0
        for row_number in range(0, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)        

        data.append(new_row)
    
df = pd.DataFrame(data, columns=columns)
    

In [7]:
# Промежуточный результат. Абсолютные значения
df

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2415690,184508,67706,229645,73773,189124,413026,319417,262743,294393,287251,481859,1933831,555632
1,2016,мужчины,1112547,94508,34693,117503,37923,94829,204901,152271,121735,123650,89304,246704,865843,284627
2,2016,женщины,1303143,90000,33013,112142,35850,94295,208125,167146,141008,170743,197947,235155,1067988,271005
3,2017,всего,2412359,183080,69747,237158,74368,176655,405187,322158,258545,294300,297552,489985,1922374,564353
4,2017,мужчины,1110187,94230,35615,121485,38178,88773,201121,153071,119665,123962,92952,251330,858857,289508
5,2017,женщины,1302172,88850,34132,115673,36190,87882,204066,169087,138880,170338,204600,238655,1063517,274845
6,2018,всего,2408221,177110,73839,243151,77090,168455,390490,328096,257544,292823,308283,494100,1914121,571190
7,2018,мужчины,1106926,91120,37669,124728,39169,85109,193483,156186,119258,123602,96859,253517,853409,292686
8,2018,женщины,1301295,85990,36170,118423,37921,83346,197007,171910,138286,169221,211424,240583,1060712,278504
9,2019,всего,2402358,170405,74605,250685,79353,165456,371416,333657,258989,287405,318048,495695,1906663,575048


In [8]:
df_avg = {}
columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]
data = []

for row_number in range(0, df.shape[0] - 3):
    new_row = []
    new_row.append(df.iloc[row_number, 0])
    new_row.append(df.iloc[row_number, 1])
    
    for col_number in range(2,df.shape[1]):
        value = (df.iloc[row_number, col_number] + df.iloc[row_number + 3, col_number])/2
        new_row.append(value)
    data.append(new_row)
df_avg = pd.DataFrame(data, columns=columns)
display(df_avg)

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2414024.5,183794.0,68726.5,233401.5,74070.5,182889.5,409106.5,320787.5,260644.0,294346.5,292401.5,485922.0,1928102.5,559992.5
1,2016,мужчины,1111367.0,94369.0,35154.0,119494.0,38050.5,91801.0,203011.0,152671.0,120700.0,123806.0,91128.0,249017.0,862350.0,287067.5
2,2016,женщины,1302657.5,89425.0,33572.5,113907.5,36020.0,91088.5,206095.5,168116.5,139944.0,170540.5,201273.5,236905.0,1065752.5,272925.0
3,2017,всего,2410290.0,180095.0,71793.0,240154.5,75729.0,172555.0,397838.5,325127.0,258044.5,293561.5,302917.5,492042.5,1918247.5,567771.5
4,2017,мужчины,1108556.5,92675.0,36642.0,123106.5,38673.5,86941.0,197302.0,154628.5,119461.5,123782.0,94905.5,252423.5,856133.0,291097.0
5,2017,женщины,1301733.5,87420.0,35151.0,117048.0,37055.5,85614.0,200536.5,170498.5,138583.0,169779.5,208012.0,239619.0,1062114.5,276674.5
6,2018,всего,2405289.5,173757.5,74222.0,246918.0,78221.5,166955.5,380953.0,330876.5,258266.5,290114.0,313165.5,494897.5,1910392.0,573119.0
7,2018,мужчины,1105026.5,89447.5,37977.5,126669.5,39602.5,84422.0,188692.5,157573.5,119334.0,122587.5,98635.0,254094.5,850932.0,293697.0
8,2018,женщины,1300263.0,84310.0,36244.5,120248.5,38619.0,82533.5,192260.5,173303.0,138932.5,167526.5,214530.5,240803.0,1059460.0,279422.0


In [9]:
path = 'E:/IrkutskProject/Data/'
df_avg.to_csv(f'{path}Common_2016-2018/avg_population_2016-2018.csv', index=False)